# 04 — Price Impact Analysis

This notebook extracts MFG price impact curves from the **OptimalExecutionMFG**
model, compares them to the empirical **square-root law**, fits the power-law
exponent δ (ΔS ∝ Q^δ), and examines how impact varies with model parameters.

### Key formulas

| Quantity | Expression |
|----------|-----------|
| Aggregate flow | M(t) = ∫ ν*(t,q) m(t,q) dq |
| Temporary impact | I_temp(t) = η · M(t) |
| Permanent impact | ΔS(Q) = η · ∫₀ᵀ M(t) dt |
| Square-root law (empirical) | ΔS ≈ σ √(Q/V) |
| Power-law fit | ΔS ≈ C · Q^δ |


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

import numpy as np
import warnings
warnings.filterwarnings("ignore")

from mfglob.grids import Grid1D, TimeGrid
from mfglob.models.optimal_execution import OptimalExecutionMFG
from mfglob.mfg_solver import MFGSolver
from mfglob.price_impact import PriceImpactAnalyzer


## 1. Solve OptimalExecution MFG

In [ ]:
model  = OptimalExecutionMFG(phi=0.5, psi=1.0, eta=0.2, lam=1.0,
                             sigma=0.05, Q0=1.0)
grid   = Grid1D(0.0, 1.5, 50)
tgrid  = TimeGrid(0.5, 50)
solver = MFGSolver(model, grid, tgrid, damping=0.5, tol=1e-3, max_iterations=40)
sol    = solver.solve(verbose=False)
pia    = PriceImpactAnalyzer(sol, model, grid, tgrid)

print(f"Converged: {sol['converged']}  ({sol['n_iterations']} iters)")


## 2. Temporary and permanent impact

In [ ]:
ti = pia.temporary_impact()    # shape (n_t,)
pi = pia.permanent_impact()    # scalar

print(f"Temporary impact profile  (first 10 steps):")
print("  t_idx   I_temp(t)")
for i in range(0, min(10, len(ti))):
    print(f"  {i:4d}    {ti[i]:.6f}")
print(f"\nPermanent impact ΔS = {pi:.6f}")
print(f"Kyle's lambda       = {pia.kyle_lambda():.6f}")


## 3. Impact curve ΔS(Q)

In [ ]:
Qs    = np.linspace(0.1, 1.8, 10)
curve = pia.impact_curve(Qs)
perms = curve['permanent_impact']
temps = curve['temporary_impact_peak']

print(f"{'Q':>8}  {'ΔS(Q)':>12}  {'I_temp_peak':>14}")
print("-" * 38)
for q, pi_q, ti_q in zip(Qs, perms, temps):
    print(f"{q:>8.3f}  {pi_q:>12.6f}  {ti_q:>14.6f}")


## 4. Power-law fit and square-root law comparison

In [ ]:
result = pia.compare_to_square_root_law(Qs)
delta  = result['mfg_exponent']
C      = result['mfg_coefficient']
sqrl   = result['sqrt_law']

print(f"Fitted power law:   ΔS ≈ {C:.5f} · Q^{delta:.3f}")
print(f"Square-root law:    ΔS ∝ Q^0.5")
print(f"MFG exponent δ  =   {delta:.3f}")
print()
print("Comparison at selected Q values:")
print(f"{'Q':>8}  {'MFG ΔS':>12}  {'Sqrt-law ΔS':>14}  {'ratio':>8}")
print("-" * 46)
for q, pi_q, sq in zip(Qs[::2], perms[::2], sqrl[::2]):
    ratio = pi_q / sq if sq > 1e-14 else float('nan')
    print(f"{q:>8.3f}  {pi_q:>12.6f}  {sq:>14.6f}  {ratio:>8.3f}")


## 5. Sensitivity: η and φ

Higher η amplifies permanent impact.  Higher φ forces faster (and thus more impactful) early execution.

In [ ]:
print("Varying η (permanent impact coefficient):")
print(f"{'η':>8}  {'exponent δ':>12}  {'Kyle λ':>10}")
print("-" * 34)
Qs_sens = np.linspace(0.15, 1.5, 8)
for eta in [0.1, 0.2, 0.3, 0.4]:
    m2 = OptimalExecutionMFG(phi=0.5, psi=1.0, eta=eta, sigma=0.05, Q0=1.0)
    g2 = Grid1D(0.0, 1.5, 40)
    t2 = TimeGrid(0.5, 40)
    s2 = MFGSolver(m2, g2, t2, damping=0.5, tol=1e-3, max_iterations=30).solve(verbose=False)
    p2 = PriceImpactAnalyzer(s2, m2, g2, t2)
    r2 = p2.compare_to_square_root_law(Qs_sens)
    lam2 = p2.kyle_lambda()
    print(f"{eta:>8.2f}  {r2['mfg_exponent']:>12.3f}  {lam2:>10.6f}")

print("\nObservation: exponent ≈ 1 for this model (linear in Q), unlike empirical ≈ 0.5.")
print("The nonlinear √Q regime emerges in models with inventory diffusion.")


## 6. Discussion

The MFG permanent impact in OptimalExecutionMFG is **approximately linear** in Q
(exponent δ ≈ 1), whereas empirical markets show the **square-root law** (δ ≈ 0.5).

The difference arises because:
- In our model, σ is small and inventory trajectories are nearly deterministic.
- The square-root law emerges from *stochastic execution* (random order flow)
  or from *market-maker inventory diffusion* coupling into the depth profile.
- Cardaliaguet & Lehalle (2018) derive √Q impact from the MFG of market makers
  when σ is large relative to inventory.

**Practical implication:** calibrating η to match Kyle's λ at small Q gives a
first-order approximation; the full impact curve requires either a larger σ or
a coupled LOB-formation model.
